In [11]:
import os
import gc
import torch
import time  # [추가] 시간 측정을 위한 내장 모듈
from llama_index.core import (
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    Settings, 
    StorageContext, 
    load_index_from_storage
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ==========================================
# [추가] 메모리 정리 유틸리티 함수
# ==========================================
def clear_memory():
    """RAM과 VRAM의 찌꺼기 메모리를 강제로 청소합니다."""
    gc.collect() # RAM 청소
    if torch.cuda.is_available():
        torch.cuda.empty_cache() # VRAM 캐시 청소
        torch.cuda.ipc_collect() # Windows 환경 VRAM 누수 방지

# 전체 실행 시간 측정 시작
total_start_time = time.time()

# 시작 시 혹시 모를 찌꺼기 메모리 초기화
clear_memory()
print("[시스템] 시작 전 메모리 초기화 완료.")

# ==========================================
# 1. 환경 설정 (LLM & 임베딩 모델)
# ==========================================
step1_start = time.time()

# Ollama 모델 파라미터에 keep_alive="0"을 주면, 답변이 끝난 직후 VRAM에서 즉시 방뺌
Settings.llm = Ollama(
    model="hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M", 
    request_timeout=360.0,
    additional_kwargs={"keep_alive": 0} # [핵심] Ollama VRAM 즉시 해제 옵션
)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3", device="cuda")

step1_end = time.time()
print(f"[분석] 1. 환경 설정 소요 시간: {step1_end - step1_start:.2f}초")

# ==========================================
# 2. 데이터 저장소(인덱스) 관리
# ==========================================
step2_start = time.time()

PERSIST_DIR = "./f1_storage"
DATA_DIR = r"D:\code\F1_Team_Radio\Related materials"

if not os.path.exists(PERSIST_DIR):
    print(f"[{PERSIST_DIR}] 폴더가 없습니다. 문서를 처음으로 임베딩합니다. (시간 소요)")
    documents = SimpleDirectoryReader(DATA_DIR).load_data()
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    
    # 임베딩이 끝났으므로 원본 원시 데이터 메모리에서 삭제
    del documents 
    print("임베딩 및 저장이 완료되었습니다.")
else:
    print(f"[{PERSIST_DIR}] 폴더를 발견했습니다. 저장된 인덱스를 로드합니다. (1초 컷)")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

step2_end = time.time()
print(f"[분석] 2. 데이터 인덱싱/로드 소요 시간: {step2_end - step2_start:.2f}초")

# ==========================================
# 3. 질문 및 답변 (RAG)
# ==========================================
step3_start = time.time()
print("\n답변을 생성 중입니다...")

query_engine = index.as_query_engine()
response = query_engine.query("""
[Role]
당신은 F1 수석 레이스 엔지니어입니다. 제공된 텔레메트리(JSON) 데이터와 드라이버의 무전을 바탕으로 전략을 즉각적으로 지시하십시오. 무조건 자연스러운 한국어로 답변해야 합니다.

[Strict Rules]
1. 어조: 단호하고 짧은 '무전(Radio)' 톤. 존댓말이나 친절한 설명, 인사말 절대 금지.
2. 금지어: "라프", "시스템을 푼다" 등의 어색한 직역 금지. 대신 "랩", "푸시해" 등의 F1 용어 사용.
3. 데이터 판단 규칙:
   - 'tire_wear_pct'가 "위험"이면: 이번 랩에 무조건 "Box" 지시.
   - 드라이버나 데이터에 옐로 플래그 언급이 없다면: 트랙 상황을 지어내지 마십시오. (정상 주행 상태로 간주)
   - 'target_ahead'의 pace가 "Slower"이면: 앞차를 언더컷(Undercut) 한다고 통보.

[Output Format]
반드시 3문장 이내로 짧게 대답하십시오. 

[Examples - 논리 구조만 참고하고 텍스트를 그대로 복사하지 마십시오]

상황 A (타이어 위험, 앞차 느림):
- 데이터: tire_wear_pct: 위험, target_ahead: "Sainz", pace: Slower
- 드라이버: "타이어가 다 된 것 같아."
- 엔지니어: "사인츠 페이스 떨어지고 있다. 타이어 마모 한계 도달했으니 이번 랩에 박스, 박스. 언더컷 노린다."

상황 B (타이어 정상, 옐로 플래그 발생):
- 드라이버: "옐로 플래그! 어떻게 해?"
- 엔지니어: "옐로 플래그 확인. 델타 타임 양수로 유지해. 타이어 온도 정상이니 트랙 클리어될 때까지 대기한다."

[Telemetry Data]
{
  "car_status": {
    "tire_cliff_predicted_lap": 18,
    "tire_wear_pct": "안전",
    "tire_temp": "Optimal"
  },
  "strategy_calculation": {
    "target_ahead": {"name": "Leclerc", "gap_sec": "약간 멀리 있음", "pace_trend": "느림"},
    "pit_window_status": "OPEN",
    "pit_exit_traffic": "Clear"
  }
}

[Driver Radio]
'오버테이크는 무슨 말이지?'

[Response]
엔지니어: 
                              """)

step3_end = time.time()

print("\n================== [레이스 엔지니어 무전] ==================")
print(response)
print("============================================================\n")
print(f"[분석] 3. 질문 및 답변 생성(추론) 소요 시간: {step3_end - step3_start:.2f}초")

# ==========================================
# [추가] 실행 종료 후 메모리 완벽 정리
# ==========================================
step4_start = time.time()

# 무거운 객체들을 메모리에서 명시적으로 삭제
del query_engine
del index
del Settings.embed_model
del Settings.llm

# 최종 메모리 청소 실행
clear_memory()

step4_end = time.time()
print(f"[분석] 4. 메모리 정리 소요 시간: {step4_end - step4_start:.2f}초")

total_end_time = time.time()
print(f"[분석] 파이프라인 총 소요 시간: {total_end_time - total_start_time:.2f}초")
print("[시스템] 실행 종료. RAM 및 VRAM 정리 완료.")

[시스템] 시작 전 메모리 초기화 완료.
[분석] 1. 환경 설정 소요 시간: 6.24초
[./f1_storage] 폴더를 발견했습니다. 저장된 인덱스를 로드합니다. (1초 컷)
[분석] 2. 데이터 인덱싱/로드 소요 시간: 14.13초

답변을 생성 중입니다...

================== [레이스 엔지니어 무전] ==================
타이어 온도가 정상이며,  라프 타임 문제 없다면 이번 라프를 전후 푸시해. 




[분석] 3. 질문 및 답변 생성(추론) 소요 시간: 9.63초


AttributeError: property 'embed_model' of '_Settings' object has no deleter

In [ ]:
import os
import gc
import torch
import time  # 시간 측정을 위한 내장 모듈
from llama_index.core import (
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    Settings, 
    StorageContext, 
    load_index_from_storage,
    PromptTemplate  # [추가] 정석 RAG 프롬프트 구성을 위한 클래스
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# ==========================================
# 메모리 정리 유틸리티 함수
# ==========================================
def clear_memory():
    """RAM과 VRAM의 찌꺼기 메모리를 강제로 청소합니다."""
    gc.collect() # RAM 청소
    if torch.cuda.is_available():
        torch.cuda.empty_cache() # VRAM 캐시 청소
        torch.cuda.ipc_collect() # Windows 환경 VRAM 누수 방지

# 전체 실행 시간 측정 시작
total_start_time = time.time()

# 시작 시 혹시 모를 찌꺼기 메모리 초기화
clear_memory()
print("[시스템] 시작 전 메모리 초기화 완료.")

# ==========================================
# 1. 환경 설정 (LLM & 임베딩 모델)
# ==========================================
step1_start = time.time()

# Ollama 모델 파라미터에 keep_alive="0"을 주면, 답변이 끝난 직후 VRAM에서 즉시 방뺌
Settings.llm = Ollama(
    model="hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M", 
    request_timeout=360.0,
    additional_kwargs={"keep_alive": 0} # [핵심] Ollama VRAM 즉시 해제 옵션
)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3", device="cuda")

step1_end = time.time()
print(f"[분석] 1. 환경 설정 소요 시간: {step1_end - step1_start:.2f}초")

# ==========================================
# 2. 데이터 저장소(인덱스) 관리
# ==========================================
step2_start = time.time()

PERSIST_DIR = "./f1_storage"
DATA_DIR = r"D:\code\F1_Team_Radio\Related materials"

if not os.path.exists(PERSIST_DIR):
    print(f"[{PERSIST_DIR}] 폴더가 없습니다. 문서를 처음으로 임베딩합니다. (시간 소요)")
    documents = SimpleDirectoryReader(DATA_DIR).load_data()
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    
    # 임베딩이 끝났으므로 원본 원시 데이터 메모리에서 삭제
    del documents 
    print("임베딩 및 저장이 완료되었습니다.")
else:
    print(f"[{PERSIST_DIR}] 폴더를 발견했습니다. 저장된 인덱스를 로드합니다. (1초 컷)")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

step2_end = time.time()
print(f"[분석] 2. 데이터 인덱싱/로드 소요 시간: {step2_end - step2_start:.2f}초")

# ==========================================
# 3. 질문 및 답변 (RAG 프롬프트 및 실행 세팅)
# ==========================================
step3_start = time.time()
print("\n답변을 생성 중입니다...")

# [수정] 템플릿을 사용하여 저장소 문서({context_str})와 실시간 쿼리({query_str}) 영역을 엄격히 격리
qa_prompt_tmpl_str = """
[Role]
당신은 F1 수석 레이스 엔지니어입니다. 오직 [실시간 상황]의 JSON 데이터만을 분석하여, 아래의 <판단 로직>을 거친 후 <출력 템플릿>에 맞춰 드라이버에게 반말로 무전을 치십시오.

[참고 문서]
{context_str}

[실시간 상황]
{query_str}

<판단 로직 및 출력 템플릿>
아래 조건 중 현재 데이터에 맞는 1개를 선택하여, 괄호 [ ] 안의 내용을 채워 출력하시오. 다른 부가 설명은 절대 하지 마시오.

조건 1: tire_wear_pct 가 "위험"인 경우
출력: "타이어 한계다. 이번 랩 박스, 박스."

조건 2: tire_wear_pct 가 "안전"이고, target_ahead 의 pace_trend 가 "느림"인 경우
출력: "타이어 정상. [target_ahead의 name] 페이스 떨어지고 있다. 언더컷 노린다. 푸시해."

조건 3: 드라이버 무전에 "옐로 플래그"가 언급된 경우
출력: "옐로 플래그 확인. 타이어 온도 [tire_temp]. 델타 타임 유지하며 대기하라."

조건 4: 위 3가지에 해당하지 않는 일반 주행 상황인 경우
출력: "카피. 타이어 [tire_wear_pct]. 현재 페이스 유지해."

엔지니어: """
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

# 쿼리 엔진 생성 및 프롬프트 주입 수락 설정
query_engine = index.as_query_engine()
query_engine.update_prompts({"response_synthesizer:text_qa_template": qa_prompt_tmpl})

# [수정] query() 내부에는 실시간 가변 데이터만 주입하여, 이 텍스트를 기반으로 문서 검색이 유도되도록 함
live_situation = """
[Telemetry Data]
{
  "car_status": {
    "tire_cliff_predicted_lap": 18,
    "tire_wear_pct": "안전",
    "tire_temp": "Optimal"
  },
  "strategy_calculation": {
    "target_ahead": {"name": "Leclerc", "gap_sec": "약간 멀리 있음", "pace_trend": "+3.4"},
    "pit_window_status": "OPEN",
    "pit_exit_traffic": "Clear"
  }
}

[Driver Radio]
'어떤 전락으로 갈까?'
"""

response = query_engine.query(live_situation)
print(live_situation)
step3_end = time.time()

print("\n================== [레이스 엔지니어 무전] ==================")
print(response)
print("============================================================\n")
print(f"[분석] 3. 질문 및 답변 생성(추론) 소요 시간: {step3_end - step3_start:.2f}초")

# ==========================================
# 4. 실행 종료 후 메모리 완벽 정리 (오류 해결)
# ==========================================
step4_start = time.time()

# 일반 변수 객체 삭제
del query_engine
del index

# [수정] AttributeError 제거를 위해 del 대신 None 대입 처리
Settings.embed_model = None
Settings.llm = None

# 최종 메모리 청소 실행
clear_memory()

step4_end = time.time()
print(f"[분석] 4. 메모리 정리 소요 시간: {step4_end - step4_start:.2f}초")

total_end_time = time.time()
print(f"[분석] 파이프라인 총 소요 시간: {total_end_time - total_start_time:.2f}초")
print("[시스템] 실행 종료. RAM 및 VRAM 정리 완료.")

[시스템] 시작 전 메모리 초기화 완료.
[분석] 1. 환경 설정 소요 시간: 6.02초
[./f1_storage] 폴더를 발견했습니다. 저장된 인덱스를 로드합니다. (1초 컷)
[분석] 2. 데이터 인덱싱/로드 소요 시간: 13.54초

답변을 생성 중입니다...

[Telemetry Data]
{
  "car_status": {
    "tire_cliff_predicted_lap": 18,
    "tire_wear_pct": "안전",
    "tire_temp": "Optimal"
  },
  "strategy_calculation": {
    "target_ahead": {"name": "Leclerc", "gap_sec": "약간 멀리 있음", "pace_trend": "+3.4"},
    "pit_window_status": "OPEN",
    "pit_exit_traffic": "Clear"
  }
}

[Driver Radio]
'어떤 전락으로 갈까?'


================== [레이스 엔지니어 무전] ==================
"타이어 온도 안전. [target_ahead의 name] 페이스 떨어지고 있다. 언더컷 노린다. 푸시해." 


[분석] 3. 질문 및 답변 생성(추론) 소요 시간: 8.79초
Embeddings have been explicitly disabled. Using MockEmbedding.
LLM is explicitly disabled. Using MockLLM.
[분석] 4. 메모리 정리 소요 시간: 0.30초
[분석] 파이프라인 총 소요 시간: 28.80초
[시스템] 실행 종료. RAM 및 VRAM 정리 완료.


In [3]:
import os
import gc
import torch
import time  # 시간 측정을 위한 내장 모듈
from llama_index.core import (
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    Settings, 
    StorageContext, 
    load_index_from_storage,
    PromptTemplate  # [추가] 정석 RAG 프롬프트 구성을 위한 클래스
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# ==========================================
# 메모리 정리 유틸리티 함수
# ==========================================
def clear_memory():
    """RAM과 VRAM의 찌꺼기 메모리를 강제로 청소합니다."""
    gc.collect() # RAM 청소
    if torch.cuda.is_available():
        torch.cuda.empty_cache() # VRAM 캐시 청소
        torch.cuda.ipc_collect() # Windows 환경 VRAM 누수 방지

# 전체 실행 시간 측정 시작
total_start_time = time.time()

# 시작 시 혹시 모를 찌꺼기 메모리 초기화
clear_memory()
print("[시스템] 시작 전 메모리 초기화 완료.")

# ==========================================
# 1. 환경 설정 (LLM & 임베딩 모델)
# ==========================================
step1_start = time.time()

# Ollama 모델 파라미터에 keep_alive="0"을 주면, 답변이 끝난 직후 VRAM에서 즉시 방뺌
Settings.llm = Ollama(
    model="hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M", 
    request_timeout=360.0,
    additional_kwargs={"keep_alive": 0} # [핵심] Ollama VRAM 즉시 해제 옵션
)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3", device="cuda")

step1_end = time.time()
print(f"[분석] 1. 환경 설정 소요 시간: {step1_end - step1_start:.2f}초")

# ==========================================
# 2. 데이터 저장소(인덱스) 관리
# ==========================================
step2_start = time.time()

PERSIST_DIR = "./f1_storage"
DATA_DIR = r"D:\code\F1_Team_Radio\Related materials"

if not os.path.exists(PERSIST_DIR):
    print(f"[{PERSIST_DIR}] 폴더가 없습니다. 문서를 처음으로 임베딩합니다. (시간 소요)")
    documents = SimpleDirectoryReader(DATA_DIR).load_data()
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    
    # 임베딩이 끝났으므로 원본 원시 데이터 메모리에서 삭제
    del documents 
    print("임베딩 및 저장이 완료되었습니다.")
else:
    print(f"[{PERSIST_DIR}] 폴더를 발견했습니다. 저장된 인덱스를 로드합니다. (1초 컷)")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

step2_end = time.time()
print(f"[분석] 2. 데이터 인덱싱/로드 소요 시간: {step2_end - step2_start:.2f}초")

# ==========================================
# 3. 질문 및 답변 (프롬프트 수동 조립 및 검증)
# ==========================================
print("\n[시스템] 데이터 검색 및 프롬프트 조립을 시작합니다...")
step3_start = time.time()

# 1. 문서 검색 (Retriever)
retriever = index.as_retriever(similarity_top_k=2)
retrieved_nodes = retriever.retrieve("F1 타이어 마모 전략, 언더컷, 옐로 플래그 대처법")

# 2. 검색된 문서(Node)들의 텍스트 결합
context_str = "\n\n".join([node.get_content() for node in retrieved_nodes])

# 3. 실시간 상황 (입력 데이터)
live_situation = """
[Telemetry Data]
{
  "car_status": {
    "tire_cliff_predicted_lap": 18,
    "tire_wear_pct": "안전",
    "tire_temp": "Optimal"
  },
  "strategy_calculation": {
    "target_ahead": {"name": "Leclerc", "gap_sec": "약간 멀리 있음", "pace_trend": "빠름"},
    "pit_window_status": "OPEN",
    "pit_exit_traffic": "Clear"
  }
}

[Driver Radio]
'어떤 전략으로 갈까?'
"""

# [추가됨] 누락되었던 시스템 프롬프트 뼈대 변수 선언
qa_prompt_tmpl_str = """
[Role]
당신은 F1 수석 레이스 엔지니어입니다. 오직 [실시간 상황]의 JSON 데이터만을 분석하여, 아래의 <판단 로직>을 거친 후 <출력 템플릿>에 맞춰 드라이버에게 반말로 무전을 치십시오.

[참고 문서]
{context_str}

[실시간 상황]
{query_str}

<판단 로직 및 출력 템플릿>
아래 조건 중 현재 데이터에 맞는 1개를 선택하여, 괄호 [ ] 안의 내용을 채워 출력하시오.

조건 1: tire_wear_pct 가 "위험"인 경우
출력: "타이어 한계다. 이번 랩 박스, 박스."

조건 2: tire_wear_pct 가 "안전"이고, target_ahead 의 pace_trend 가 "느림" 또는 양수(+)인 경우
출력: "타이어 정상. [target_ahead의 name] 페이스 떨어지고 있다. 언더컷 노린다. 푸시해."

조건 3: 드라이버 무전에 "옐로 플래그"가 언급된 경우
출력: "옐로 플래그 확인. 타이어 온도 [tire_temp]. 델타 타임 유지하며 대기하라."

조건 4: 위 3가지에 해당하지 않는 일반 주행 상황인 경우
출력: "카피. 타이어 [tire_wear_pct]. 현재 페이스 유지해."

엔지니어: """

# 4. 최종 원시 프롬프트(Raw Prompt) 수동 조립
final_raw_prompt = qa_prompt_tmpl_str.format(
    context_str=context_str,
    query_str=live_situation
)

# 5. [핵심 검증] 조립된 100% 원본 텍스트를 콘솔에 출력
print("\n================== [LLM 입력 원본 프롬프트 (검증용)] ==================")
print(final_raw_prompt)
print("=======================================================================\n")

# 6. 완성된 원시 프롬프트를 LLM에 직접 주입
print("답변을 생성 중입니다...")
response = Settings.llm.complete(final_raw_prompt)

step3_end = time.time()

print("\n================== [레이스 엔지니어 무전 (최종 결과)] ==================")
print(response.text)
print("========================================================================\n")
print(f"[분석] 3. 질문 및 답변 생성(추론) 소요 시간: {step3_end - step3_start:.2f}초")

# ==========================================
# 4. 실행 종료 후 메모리 완벽 정리 (오류 해결)
# ==========================================
step4_start = time.time()

# 일반 변수 객체 삭제
del query_engine
del index

# [수정] AttributeError 제거를 위해 del 대신 None 대입 처리
Settings.embed_model = None
Settings.llm = None

# 최종 메모리 청소 실행
clear_memory()

step4_end = time.time()
print(f"[분석] 4. 메모리 정리 소요 시간: {step4_end - step4_start:.2f}초")

total_end_time = time.time()
print(f"[분석] 파이프라인 총 소요 시간: {total_end_time - total_start_time:.2f}초")
print("[시스템] 실행 종료. RAM 및 VRAM 정리 완료.")

[시스템] 시작 전 메모리 초기화 완료.
[분석] 1. 환경 설정 소요 시간: 15.65초
[./f1_storage] 폴더를 발견했습니다. 저장된 인덱스를 로드합니다. (1초 컷)
[분석] 2. 데이터 인덱싱/로드 소요 시간: 15.78초

[시스템] 데이터 검색 및 프롬프트 조립을 시작합니다...

================== [LLM 입력 원본 프롬프트 (검증용)] ==================

[Role]
당신은 F1 수석 레이스 엔지니어입니다. 오직 [실시간 상황]의 JSON 데이터만을 분석하여, 아래의 <판단 로직>을 거친 후 <출력 템플릿>에 맞춰 드라이버에게 반말로 무전을 치십시오.

[참고 문서]
75 501.63 540.75 520.38]
/A <</Type /Action
/S /URI
/URI (https://www.pirelli.com/tire/us/en/news/2017/11/23/pirelli%E2%80%99s-rainbow-colored-tires-will-light-up-formula-1-in-2018/)>>>>
endobj
365 0 obj
<</Type /Annot
/Subtype /Link
/F 4
/Border [0 0 0]
/Rect [60.75 483.63 551.25 502.38]
/A <</Type /Action
/S /URI
/URI (https://www.pirelli.com/tire/us/en/news/2017/11/23/pirelli%E2%80%99s-rainbow-colored-tires-will-light-up-formula-1-in-2018/)>>>>
endobj
366 0 obj
<</Type /Annot
/Subtype /Link
/F 4
/Border [0 0 0]
/Rect [60.75 465.63 170.25 484.38]
/A <</Type /Action
/S /URI
/URI (https://www.pirelli.com/tire/us/en/news/2017/11/23/pirelli%E

NameError: name 'query_engine' is not defined